# 🥣 Kohlenhydrat-Schätzer für Mahlzeiten

Eingabe: eine **verbale** Mahlzeitenbeschreibung (Freitext, Deutsch).  
Ausgabe: die geschätzten **Kohlenhydrate in Gramm**.

## Architektur (Domain Driven Design)

Wie ein Restaurant-Team:

| DDD-Begriff | Hier konkret | Restaurant-Analogie |
|---|---|---|
| Value Object | `Carbohydrates`, `Ingredient` | Posten auf der Rechnung |
| Aggregate Root | `Meal` | Die ganze Bestellung |
| Domain Service | `MealAnalyzer` | Der Ernährungsberater |
| Infrastructure | Anthropic-API | Sein Nachschlagewerk |

Die Schätzung selbst (Mengen + Kohlenhydrate) übernimmt das Claude-Modell, weil natürliche Sprache wie *"ein Viertel Apfel"* schwer mit Regeln zu parsen ist.

## 1. Setup

Falls noch nicht installiert:

In [ ]:
# %pip install anthropic

In [1]:
import os
import json
from dotenv import load_dotenv
from dataclasses import dataclass, field
from typing import List
from anthropic import Anthropic

# API-Key setzen (entweder hier oder über Umgebungsvariable ANTHROPIC_API_KEY)
load_dotenv()
# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'

client = Anthropic()  # liest automatisch ANTHROPIC_API_KEY

## 2. Domain Model

Reine Datenstrukturen ohne Abhängigkeiten — das ist der Kern, den du auch ohne API testen könntest.

In [2]:
@dataclass(frozen=True)
class Carbohydrates:
    """Wertobjekt: Kohlenhydratmenge in Gramm. Unveränderlich, addierbar."""
    grams: float

    def __add__(self, other: "Carbohydrates") -> "Carbohydrates":
        return Carbohydrates(self.grams + other.grams)

    def __radd__(self, other):
        # damit sum(...) funktioniert
        if other == 0:
            return self
        return self.__add__(other)

    def __str__(self) -> str:
        return f"{self.grams:.1f} g"


@dataclass(frozen=True)
class Ingredient:
    """Wertobjekt: eine einzelne Zutat mit Schätzungen."""
    name: str
    estimated_weight_g: float
    carbs: Carbohydrates


@dataclass
class Meal:
    """Aggregate Root: eine Mahlzeit besteht aus Zutaten."""
    description: str
    ingredients: List[Ingredient] = field(default_factory=list)

    @property
    def total_carbs(self) -> Carbohydrates:
        return sum(i.carbs for i in self.ingredients) or Carbohydrates(0)

    def report(self) -> str:
        lines = [f"Mahlzeit: {self.description}", "-" * 50]
        for ing in self.ingredients:
            lines.append(
                f"  • {ing.name:<30} "
                f"{ing.estimated_weight_g:>6.1f} g  →  {ing.carbs}"
            )
        lines.append("-" * 50)
        lines.append(f"  Gesamt-Kohlenhydrate: {self.total_carbs}")
        return "\n".join(lines)

## 3. Domain Service: `MealAnalyzer`

Hier ruft das Claude-Modell für jede Zutat eine Schätzung ab. Wichtig: das Modell antwortet **strikt im JSON-Format**, damit wir es deterministisch parsen können.

In [3]:
SYSTEM_PROMPT = """Du bist ein Ernährungsexperte. Du erhältst eine umgangssprachliche \
Beschreibung einer Mahlzeit auf Deutsch und gibst für jede Zutat zurück:
  - name (string, deutsch)
  - gewicht_g (float, geschätztes Gewicht in Gramm)
  - kohlenhydrate_g (float, Kohlenhydrate in Gramm)

Faustregeln für übliche Mengen:
  - 1 Suppenlöffel (Esslöffel) Trockenes (Haferflocken, Mehl) ≈ 10–15 g
  - 1 Teelöffel ≈ 5 g
  - 1 Tasse ≈ 200–250 ml
  - mittelgroßer Apfel ≈ 180 g, Banane ≈ 120 g, Pfirsich ≈ 150 g
  - 1 Scheibe Brot ≈ 30 g

Antworte AUSSCHLIESSLICH mit gültigem JSON in genau diesem Format \
(kein Markdown, keine Erklärung, kein Vorwort):
{
  "zutaten": [
    {"name": "...", "gewicht_g": 0.0, "kohlenhydrate_g": 0.0}
  ]
}"""


class MealAnalyzer:
    """Domain Service: übersetzt freien Text in ein strukturiertes Meal."""

    def __init__(self, client: Anthropic, model: str = "claude-sonnet-4-5"):
        self.client = client
        self.model = model

    def analyze(self, description: str) -> Meal:
        response = self.client.messages.create(
            model=self.model,
            max_tokens=1024,
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": description}],
        )

        raw = response.content[0].text.strip()
        # Falls das Modell doch mal Markdown-Fences setzt, vorsichtshalber entfernen:
        if raw.startswith("```"):
            raw = raw.strip("`")
            if raw.lower().startswith("json"):
                raw = raw[4:].strip()

        data = json.loads(raw)

        ingredients = [
            Ingredient(
                name=z["name"],
                estimated_weight_g=float(z["gewicht_g"]),
                carbs=Carbohydrates(grams=float(z["kohlenhydrate_g"])),
            )
            for z in data["zutaten"]
        ]
        return Meal(description=description, ingredients=ingredients)

## 4. Verwendung

Dein Beispiel:

In [4]:
analyzer = MealAnalyzer(client)

beschreibung = "2 Suppenlöffel Haferflocken, ein Viertel Apfel, eine halbe Banane, ein halber Pfirsich"

mahlzeit = analyzer.analyze(beschreibung)
print(mahlzeit.report())

Mahlzeit: 2 Suppenlöffel Haferflocken, ein Viertel Apfel, eine halbe Banane, ein halber Pfirsich
--------------------------------------------------
  • Haferflocken                     25.0 g  →  16.2 g
  • Apfel                            45.0 g  →  5.8 g
  • Banane                           60.0 g  →  13.2 g
  • Pfirsich                         75.0 g  →  7.5 g
--------------------------------------------------
  Gesamt-Kohlenhydrate: 42.8 g


### Nur die Zahl ausgeben

In [ ]:
print(f"Kohlenhydrate: {mahlzeit.total_carbs.grams:.1f} g")

## 5. Weitere Beispiele zum Ausprobieren

In [ ]:
beispiele = [
    "eine Scheibe Vollkornbrot mit Butter und Marmelade",
    "100 g Spaghetti mit Tomatensoße",
    "ein Joghurt mit einer Handvoll Heidelbeeren und einem Teelöffel Honig",
]

for b in beispiele:
    m = analyzer.analyze(b)
    print(m.report())
    print()

## 6. Erweiterungsideen

- **Eiweiß und Fett** zusätzlich schätzen (neue Value Objects, gleiche Architektur).
- **Lokale Datenbank** (z. B. BLS oder USDA) als zweite Infrastructure-Implementierung — der Domain Service bleibt unverändert. Das ist genau der Punkt von DDD: Schätzungslogik austauschen ohne Domain anzufassen.
- **Validierung**: Plausibilitätsprüfung (z. B. Kohlenhydrate dürfen nicht > Gewicht sein).
- **Streaming**: bei langen Mahlzeiten mit `client.messages.stream(...)` arbeiten.